In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, roc_curve
import joblib
import warnings
warnings.filterwarnings('ignore')

# Ustawienia wizualizacji
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Wczytanie danych
df_jfif = pd.read_csv('results_jfif.csv')
df_ppm = pd.read_csv('results_ppm.csv')

# Dodanie etykiet
df_jfif['label'] = 'jfif'
df_ppm['label'] = 'ppm'

# Połączenie danych
df = pd.concat([df_jfif, df_ppm], ignore_index=True)

print(f"Łączna liczba próbek: {len(df)}")
print(f"Liczba próbek JFIF: {len(df_jfif)}")
print(f"Liczba próbek PPM: {len(df_ppm)}")
print(f"\nKształt danych: {df.shape}")

# Wybór cech (bez filename i extension)
feature_columns = ['blockiness_0', 'blockiness_1', 'blockiness_2', 'blockiness_3', 
                   'blockiness_4', 'blockiness_5', 'blockiness_6', 'blockiness_7',
                   'blockiness_max', 'blockiness_mean', 'blockiness_std',
                   'energy_mean', 'energy_std', 'estimated_quality', 'psnr']

X = df[feature_columns]
y = df['label']

# Konwersja etykiet na wartości binarne (0 = jfif, 1 = ppm)
y_binary = (y == 'ppm').astype(int)

# Podział na zbiór treningowy, walidacyjny i testowy z losowym mieszaniem
# Najpierw oddzielamy zbiór testowy (20%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y_binary, test_size=0.2, random_state=42, stratify=y_binary, shuffle=True)

# Następnie dzielimy pozostałe dane na treningowy (64%) i walidacyjny (16%)
# 0.2 z 0.8 = 0.16 całości
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp, shuffle=True)

X_test.head()

Łączna liczba próbek: 22338
Liczba próbek JFIF: 11000
Liczba próbek PPM: 11338

Kształt danych: (22338, 18)


,blockiness_0,blockiness_1,blockiness_2,blockiness_3,blockiness_4,blockiness_5,blockiness_6,blockiness_7,blockiness_max,blockiness_mean,blockiness_std,energy_mean,energy_std,estimated_quality,psnr
12669,0.992951,1.016278,0.986593,1.012158,0.992269,1.001143,0.994625,1.004133,1.016278,1.000019,0.009678,171.911255,258.614380,95,46.257703
13416,0.991766,0.995640,0.998726,0.999534,0.994475,1.005847,1.012399,1.001662,1.012399,1.000006,0.006223,128.419357,275.891785,95,48.300337
3042,0.989677,1.009626,0.990827,1.006095,0.972703,1.005964,0.999501,1.026002,1.026002,1.000049,0.014906,226.298157,165.680664,95,46.876860
17656,1.001001,1.001826,1.004927,1.006509,0.992172,0.999917,1.006348,0.987201,1.006509,0.999987,0.006484,233.575760,234.240234,95,44.678300
9439,0.971133,1.046585,0.980453,1.034731,0.966116,1.020356,0.962200,1.019553,1.046585,1.000141,0.031554,197.435104,207.182968,95,47.480761


In [34]:
import joblib

loaded_model = joblib.load('models/best_model_svm_(rbf).pkl')
loaded_scaler = joblib.load('models/scaler.pkl')
loaded_feature_columns = joblib.load('models/feature_columns.pkl')

print("✓ Model wczytany pomyślnie!")
print(f"✓ Typ modelu: {type(loaded_model).__name__}")
print(f"✓ Liczba cech: {len(loaded_feature_columns)}")

# Przykładowa predykcja z wczytanym modelem
# Przygotowanie danych testowych
test_sample = X_test.iloc[0:1]  # Pierwsza próbka ze zbioru testowego
# Wyśietl próbke, etykietę i nazwe jej pliku
print(f"\nPróbka testowa (plik: {df.iloc[X_test.index[0]]['filename']}):")
print(f"Etykieta: {'PPM' if y_test.iloc[0] == 1 else 'JFIF'}")
test_sample_scaled = loaded_scaler.transform(test_sample)

# Predykcja
prediction = loaded_model.predict(test_sample_scaled)[0]
probabilities = loaded_model.predict_proba(test_sample_scaled)[0]

print(f"\nPredykcja: {'PPM' if prediction == 1 else 'JFIF'}")
print(f"Prawdopodobieństwa: JFIF={probabilities[0]*100:.2f}%, PPM={probabilities[1]*100:.2f}%")

✓ Model wczytany pomyślnie!
✓ Typ modelu: SVC
✓ Liczba cech: 15

Próbka testowa (plik: 00213_940128_fb):
Etykieta: PPM

Predykcja: PPM
Prawdopodobieństwa: JFIF=39.85%, PPM=60.15%


# Test modelu na innych zbiorach

## Wczytanie pliku

In [37]:
# Ścieżka do pliku CSV ze wskaźnikami
# csv_file = 'results_RGB-faces-128x128.csv'
csv_file = 'results_thispersondoesnotexist.10k(stare).csv'
csv_file = 'multiple-compression-results/compression_analysis_results.csv'
# csv_file = 'results_the_tufts.csv'
# csv_file = 'results_jfif.csv'
# csv_file = 'results_multiple_compression_different_quality.csv'
csv_file = 'results_test_01.csv'
csv_file = 'results_240k_one_compression_filtered.csv'
csv_file = 'results_iFakeFaceDB.csv'
# csv_file = "results_the_tufts.csv"

# Wczytanie danych
df_new = pd.read_csv(csv_file)

# Wybór tylko cech używanych przez model
X_new = df_new[loaded_feature_columns]

df_new.head()

,filename,extension,blockiness_0,blockiness_1,blockiness_2,blockiness_3,blockiness_4,blockiness_5,blockiness_6,blockiness_7,blockiness_max,blockiness_mean,blockiness_std,energy_mean,energy_std,estimated_quality,psnr
0,8,.jpg,0.992666,0.992800,1.004289,0.990242,1.005639,1.021728,1.020389,0.971611,1.021728,0.999920,0.015592,238.106247,175.519073,95,62.291457
1,9,.jpg,1.022676,0.997655,0.998614,0.968157,1.012759,1.005373,1.000040,0.994788,1.022676,1.000008,0.014809,379.096313,271.296814,95,61.224648
2,10,.jpg,0.975634,0.968364,0.986628,0.978255,1.024780,1.046203,1.001363,1.020064,1.046203,1.000161,0.025979,286.234802,231.108597,95,61.827569
3,11,.jpg,0.978805,0.952967,1.003566,1.031920,1.008710,0.973042,1.014269,1.038680,1.038680,1.000244,0.027827,212.092270,208.680054,95,62.361964
4,14,.jpg,0.989128,1.030054,1.006470,0.982038,0.955791,0.996481,1.020729,1.020462,1.030054,1.000144,0.022916,344.745850,226.218231,95,62.210988


# Test pojedynczy

In [13]:
test_sample = X_new.iloc[0:1]  # Pierwsza próbka ze zbioru testowego

print(f"\nPróbka testowa (plik: {df_new.iloc[X_new.index[0]]['filename']}):")
# print(f"Etykieta: {'PPM' if df_new.iloc[0] == 1 else 'JFIF'}")
test_sample_scaled = loaded_scaler.transform(test_sample)

# Predykcja
prediction = loaded_model.predict(test_sample_scaled)[0]
probabilities = loaded_model.predict_proba(test_sample_scaled)[0]

print(f"\nPredykcja: {'PPM' if prediction == 1 else 'JFIF'}")
print(f"Prawdopodobieństwa: JFIF={probabilities[0]*100:.2f}%, PPM={probabilities[1]*100:.2f}%")


Próbka testowa (plik: 0):

Predykcja: JFIF
Prawdopodobieństwa: JFIF=99.99%, PPM=0.01%


# Test całego zbioru

In [38]:
# Sprawdzenie brakujących danych
print(f"Całkowita liczba próbek: {len(df_new)}")
missing_count = X_new.isnull().any(axis=1).sum()
print(f"Liczba wierszy z brakującymi danymi: {missing_count}")

# Usunięcie wierszy z brakującymi danymi
valid_indices = X_new.dropna().index
X_new_clean = X_new.loc[valid_indices]
df_new_clean = df_new.loc[valid_indices].copy()

print(f"Liczba próbek po usunięciu brakujących danych: {len(X_new_clean)}")

# Skalowanie wszystkich danych
X_new_scaled = loaded_scaler.transform(X_new_clean)

# Predykcja dla całego zbioru
predictions = loaded_model.predict(X_new_scaled)
probabilities = loaded_model.predict_proba(X_new_scaled)

# Dodanie wyników do DataFrame
df_new_clean['prediction'] = ['PPM' if pred == 1 else 'JFIF' for pred in predictions]
df_new_clean['probability_JFIF'] = probabilities[:, 0] * 100
df_new_clean['probability_PPM'] = probabilities[:, 1] * 100

# Statystyki
print(f"\nRozkład predykcji:")
print(df_new_clean['prediction'].value_counts())
print(f"\nProcentowy rozkład:")
print(df_new_clean['prediction'].value_counts(normalize=True) * 100)

# Wyświetlenie próbek sklasyfikowanych jako PPM
print(f"\n{'='*80}")
print("Próbki sklasyfikowane jako PPM (niekompresowany):")
print(f"{'='*80}")
ppm_samples = df_new_clean[df_new_clean['prediction'] == 'PPM']
for idx, row in ppm_samples.iterrows():
    print(f"{row['filename']}{row['extension']} - sz. jakość: {row['estimated_quality']} ,pewność: {row['probability_PPM']:.2f}%")

# Zapisanie wyników do pliku CSV
# output_file = 'predictions_RGB-faces-128x128.csv'
# df_new_clean.to_csv(output_file, index=False)
# print(f"\n{'='*80}")
# print(f"✓ Wyniki zapisane do pliku: {output_file}")
# print(f"✓ Zapisano {len(df_new_clean)} próbek (pominięto {missing_count} z brakującymi danymi)")

Całkowita liczba próbek: 62957
Liczba wierszy z brakującymi danymi: 0
Liczba próbek po usunięciu brakujących danych: 62957

Rozkład predykcji:
prediction
JFIF    62957
Name: count, dtype: int64

Procentowy rozkład:
prediction
JFIF    100.0
Name: proportion, dtype: float64

Próbki sklasyfikowane jako PPM (niekompresowany):


# Analiza pojedynczego pliku obrazu
Wskaż plik obrazu, aby obliczyć wskaźniki i dokonać predykcji

In [31]:
from index_script import detect_jpeg_compression
import os

def check_image(image_path):
    if not os.path.exists(image_path):
        print(f"Błąd: Plik '{image_path}' nie istnieje!")
        return
    
    print(f"Analiza pliku: {image_path}")
    print("-" * 80)
    
    try:
        result = detect_jpeg_compression(image_path)
        
        if result is None:
            print("Nie udało się obliczyć wskaźników")
            return
        
        # Wyświetl wskaźniki
        # print("\nObliczone wskaźniki:")
        # for key, value in result.items():
        #     if key not in ['filename', 'extension']:
        #         if isinstance(value, float):
        #             print(f"{key:20s}: {value:.4f}")
        #         else:
        #             print(f"{key:20s}: {value}")
        
        # Przygotuj dane do predykcji
        feature_dict = {col: result.get(col, 0) for col in loaded_feature_columns}
        X_single = pd.DataFrame([feature_dict])
        
        if X_single.isnull().any().any():
            print("\nBrakujące dane w wskaźnikach:")
            print(X_single.isnull().sum()[X_single.isnull().sum() > 0])
            return
        
        # Skalowanie i predykcja
        X_single_scaled = loaded_scaler.transform(X_single)
        prediction = loaded_model.predict(X_single_scaled)[0]
        probabilities = loaded_model.predict_proba(X_single_scaled)[0]
        
        # Wyniki
        print("\n" + "-" * 80)
        print("Wynik predykcji:")
        print("-" * 80)
        print(f"Format: {'PPM (niekompresowany)' if prediction == 1 else 'JFIF (kompresja JPEG)'}")
        print(f"\nPrawdopodobieństwa:")
        print(f"  JFIF:  {probabilities[0]*100:6.2f}%")
        print(f"  PPM:   {probabilities[1]*100:6.2f}%")
        
        confidence = max(probabilities) * 100
        print(f"\nPewność: {confidence:.1f}%")
        
    except Exception as e:
        print(f"Błąd: {str(e)}")
        import traceback
        traceback.print_exc()


In [33]:
# ============================
# WSKAŻ PLIK DO ANALIZY
# ============================
# image_path = 'multiple-compression/01 (5).png'  # Zmień na ścieżkę do swojego pliku
# image_path = 'ColorFeret-unpacked/00001_930831_fa_a.ppm'
# image_path = 'jfif/00001_930831_fa_a.jpg'
image_path = 'phone-images/zdjecie_zdjecia_03.dng.png'  # Przykładowy plik do analizy
image_path = r"C:\Users\01sok\Downloads\test-224-224-1do1.png"
image_path = r"C:\Users\01sok\Downloads\test-500-700q80.png"
# image_path = r"C:\Users\01sok\Downloads\test-224-224-1do1.jpg"
# image_path = r"C:\Users\01sok\Downloads\oryg-512-700.jpg"
image_path = r"C:\Users\01sok\Downloads\test-500-700q80-1do1-doPNG-224-224.png"
# image_path = r"C:\Users\01sok\Downloads\test-500-700q80-1do1-doPNG-400-400.png"
# image_path = r"C:\Users\01sok\Downloads\test-500-700q80-1do1-doPNG-500-500.png"


check_image(image_path)

check_image(r"C:\Users\01sok\Downloads\fakeface.jpg")
check_image(r"C:\Users\01sok\Downloads\fakeface-100x100.png")
check_image(r"C:\Users\01sok\Downloads\fakeface-500x500.png")
check_image(r"C:\Users\01sok\Downloads\fakeface-250x250.png")
check_image(r"C:\Users\01sok\Downloads\test-500-700q60q100-1000x1500.png")

Analiza pliku: C:\Users\01sok\Downloads\test-500-700q80-1do1-doPNG-224-224.png
--------------------------------------------------------------------------------

--------------------------------------------------------------------------------
Wynik predykcji:
--------------------------------------------------------------------------------
Format: PPM (niekompresowany)

Prawdopodobieństwa:
  JFIF:   51.84%
  PPM:    48.16%

Pewność: 51.8%
Analiza pliku: C:\Users\01sok\Downloads\fakeface.jpg
--------------------------------------------------------------------------------

--------------------------------------------------------------------------------
Wynik predykcji:
--------------------------------------------------------------------------------
Format: JFIF (kompresja JPEG)

Prawdopodobieństwa:
  JFIF:   79.63%
  PPM:    20.37%

Pewność: 79.6%
Analiza pliku: C:\Users\01sok\Downloads\fakeface-100x100.png
--------------------------------------------------------------------------------

-